# Yagye Data Lake Explorer

Query Bronze (raw NDJSON), Silver (typed Parquet), and Gold (aggregated marts) directly via DuckDB.
The connection is pre-configured to point at Floci (local S3 emulator) — identical to how the ETL works.

**Buckets**
| Layer | Bucket | Written by | Cadence |
|---|---|---|---|
| Bronze | `yagye-lake-raw` | `DataLakeIngestionWorker` (Core) | every 1 min |
| Silver | `yagye-lake-silver` | `run_silver_etl.py` | every 1 h |
| Gold | `yagye-lake-gold` | `run_gold_etl.py` | every 6 h |
| ML models | `yagye-ml-models` | `train_models.py` | manual |


In [ ]:
import os
import duckdb

# S3 / Floci config — reads from env vars set by docker-compose
S3_ENDPOINT  = os.environ.get('S3_ENDPOINT_URL', 'http://localhost:4566')
S3_KEY       = os.environ.get('AWS_ACCESS_KEY_ID', 'test')
S3_SECRET    = os.environ.get('AWS_SECRET_ACCESS_KEY', 'test')
S3_REGION    = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')

BRONZE = os.environ.get('BRONZE_BUCKET', 'yagye-lake-raw')
SILVER = os.environ.get('SILVER_BUCKET', 'yagye-lake-silver')
GOLD   = os.environ.get('GOLD_BUCKET',   'yagye-lake-gold')

# DuckDB with httpfs pointed at Floci — same setup as the ETL
con = duckdb.connect()
con.execute('INSTALL httpfs; LOAD httpfs; INSTALL parquet; LOAD parquet;')

endpoint = S3_ENDPOINT.removeprefix('https://').removeprefix('http://')
con.execute(f"""
    SET s3_endpoint           = '{endpoint}';
    SET s3_access_key_id      = '{S3_KEY}';
    SET s3_secret_access_key  = '{S3_SECRET}';
    SET s3_region             = '{S3_REGION}';
    SET s3_url_style          = 'path';
    SET s3_use_ssl            = false;
""")

print(f'DuckDB connected → S3 endpoint: {endpoint}')
print(f'Bronze: s3://{BRONZE}  Silver: s3://{SILVER}  Gold: s3://{GOLD}')

## Bronze — raw outbox events (NDJSON)

Written by `DataLakeIngestionWorker` in Core every minute. Each file is one polling window of outbox events, keyed as `events/dt=YYYY-MM-DD/<prefix>_<us>.ndjson`.

In [ ]:
# List Bronze files — what's landed so far?
import boto3

s3 = boto3.client(
    's3',
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_KEY,
    aws_secret_access_key=S3_SECRET,
    region_name=S3_REGION,
)

resp = s3.list_objects_v2(Bucket=BRONZE, Prefix='events/')
files = resp.get('Contents', [])
print(f'{len(files)} Bronze file(s) in s3://{BRONZE}/events/')
for f in files[:10]:
    print(f"  {f['Key']}  ({f['Size']:,} bytes)")
if len(files) > 10:
    print(f'  ... and {len(files)-10} more')

In [ ]:
# Peek at raw Bronze events — what event types exist?
try:
    df = con.execute(f"""
        SELECT
            event_type,
            COUNT(*)              AS count,
            MIN(occurred_at)      AS earliest,
            MAX(occurred_at)      AS latest
        FROM read_ndjson_auto(
            's3://{BRONZE}/events/dt=*/*.ndjson',
            union_by_name = true,
            ignore_errors = true
        )
        GROUP BY event_type
        ORDER BY count DESC
    """).df()
    display(df)
except Exception as e:
    print(f'No Bronze data yet (or no files matched): {e}')
    print('Make some payments at http://localhost:4010 then wait ~1 min')

In [ ]:
# Sample 5 raw payment.succeeded events — see full envelope + payload
try:
    df = con.execute(f"""
        SELECT
            event_id, event_type, merchant_id, aggregate_id AS payment_id,
            occurred_at, mode,
            json_extract_string(payload, '$.amount')        AS amount,
            json_extract_string(payload, '$.currency')      AS currency,
            json_extract_string(payload, '$.method')        AS method,
            json_extract_string(payload, '$.provider_code') AS provider_code,
            json_extract_string(payload, '$.provider_kind') AS provider_kind,
            json_extract_string(payload, '$.network')       AS network,
            json_extract(payload, '$.latency_ms')           AS latency_ms
        FROM read_ndjson_auto(
            's3://{BRONZE}/events/dt=*/*.ndjson',
            union_by_name = true, ignore_errors = true
        )
        WHERE event_type = 'payment.succeeded'
        ORDER BY occurred_at DESC
        LIMIT 5
    """).df()
    display(df)
except Exception as e:
    print(f'No payment.succeeded events yet: {e}')

## Silver — typed, deduplicated Parquet

Written by `run_silver_etl.py` hourly (or run `make silver` from `apps/yagye_data/`). Four tables: payments, settlements, merchants, reconciliation. Each partitioned by `dt=YYYY-MM-DD`.

In [ ]:
# Silver payments — the ML feature table
try:
    df = con.execute(f"""
        SELECT
            dt,
            event_type,
            provider_code,
            provider_kind,
            method,
            network,
            currency,
            amount_minor / 100.0     AS amount_ghs,
            latency_ms,
            mode
        FROM read_parquet('s3://{SILVER}/payments/dt=*/part.parquet')
        ORDER BY dt DESC
        LIMIT 20
    """).df()
    display(df)
except Exception as e:
    print(f'No Silver payments yet: {e}')
    print('Run: cd apps/yagye_data && make silver')

In [ ]:
# Silver payments — success rate by provider (the core ML training signal)
try:
    df = con.execute(f"""
        SELECT
            provider_code,
            provider_kind,
            COUNT(*)                                                AS total,
            SUM(CASE WHEN event_type='payment.succeeded' THEN 1 ELSE 0 END) AS succeeded,
            ROUND(
                SUM(CASE WHEN event_type='payment.succeeded' THEN 1 ELSE 0 END)
                * 100.0 / COUNT(*), 2
            )                                                       AS success_pct,
            ROUND(AVG(latency_ms), 0)                               AS avg_latency_ms,
            ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY latency_ms), 0) AS p50_latency_ms,
            ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY latency_ms), 0) AS p95_latency_ms
        FROM read_parquet('s3://{SILVER}/payments/dt=*/part.parquet')
        GROUP BY provider_code, provider_kind
        ORDER BY total DESC
    """).df()
    display(df)
except Exception as e:
    print(f'No Silver data: {e}')

In [ ]:
# Silver settlements
try:
    df = con.execute(f"""
        SELECT *
        FROM read_parquet('s3://{SILVER}/settlements/dt=*/part.parquet')
        ORDER BY dt DESC LIMIT 10
    """).df()
    display(df)
except Exception as e:
    print(f'No Silver settlements yet: {e}')

In [ ]:
# Silver reconciliation breaks
try:
    df = con.execute(f"""
        SELECT *
        FROM read_parquet('s3://{SILVER}/reconciliation/dt=*/part.parquet')
        ORDER BY dt DESC LIMIT 10
    """).df()
    display(df)
except Exception as e:
    print(f'No Silver reconciliation data yet: {e}')

## Gold — aggregated marts

Written by `run_gold_etl.py` every 6 hours (or `make gold`). Three marts: `payments_mart`, `risk_mart`, `ops_mart`. These are the inputs for dashboards and ML feature engineering.

In [ ]:
# Gold payments_mart — daily/hourly aggregates used in routing model features
try:
    df = con.execute(f"""
        SELECT *
        FROM read_parquet('s3://{GOLD}/payments_mart/*.parquet')
        ORDER BY 1 DESC LIMIT 20
    """).df()
    display(df)
except Exception as e:
    print(f'No Gold payments_mart yet: {e}')
    print('Run: cd apps/yagye_data && make gold  (requires Silver to exist first)')

In [ ]:
# Gold risk_mart
try:
    df = con.execute(f"""
        SELECT * FROM read_parquet('s3://{GOLD}/risk_mart/*.parquet') LIMIT 20
    """).df()
    display(df)
except Exception as e:
    print(f'No Gold risk_mart yet: {e}')

## Ad-hoc queries

Write any DuckDB SQL below — it reads directly from Floci S3.

In [ ]:
# ✏️ Your query here
sql = f"""
SELECT 'replace this with any DuckDB SQL' AS hint
"""

con.execute(sql).df()